# Phase 06B - BTUMQA-225K Task-Trained Parameter Audit

This CPU-only notebook creates the authoritative task-trained parameter counts for the
same-dataset comparison in Table VI. It does **not** train models, run inference, download
pretrained encoders, or overwrite Phase 04/05 artifacts.

Primary scope: parameters optimized for answer prediction on BTUMQA-225K. Frozen encoders,
offline feature extractors, and the post-prediction Q-CUR reliability estimator are excluded.


# Dual Environment Compatibility Setup & Install Required Libraries

In [ ]:
# ── DUAL ENVIRONMENT COMPATIBILITY & DEPENDENCY SETUP ────────────────────────
import os
import sys
from pathlib import Path

def resolve_project_environment(mount_point: str = "/content/drive") -> tuple[Path, Path]:
    try:
        import google.colab
        from google.colab import drive
        drive.mount(mount_point)
        project_root = Path(mount_point) / "MyDrive" / "AUGR-VQA"
        temp_dir = Path("/content")
        print("Running in Google Colab environment.")
    except ImportError:
        # Running locally (parent of notebooks directory)
        project_root = Path(os.getcwd()).parent.resolve()
        temp_dir = project_root / "temp"
        temp_dir.mkdir(parents=True, exist_ok=True)
        print("Running in Local environment.")
    return project_root, temp_dir

PROJECT_ROOT, TEMP_DIR = resolve_project_environment()
# ─────────────────────────────────────────────────────────────────────────────

import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    from IPython.display import display
except Exception:
    display = print

print("Torch:", torch.__version__)
print("CUDA is not required; audit device: CPU")


Torch: 2.11.0+cpu
CUDA is not required; audit device: CPU


## Google Drive and Output Paths


In [ ]:
try:
    from google.colab import drive
#     drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using the current working directory for path inspection.")


# Updated project path to point to the new drive structure
PROJECT_DRIVE_DIR = PROJECT_ROOT
if not PROJECT_DRIVE_DIR.exists():
    PROJECT_DRIVE_DIR = Path.cwd()

# Corrected paths: phase_6/p6b_model_complexity/btumqa_225k_clean_metadata_task_trained_parameter_audit
PHASE06B_DIR = (
    PROJECT_DRIVE_DIR
    / "phase_6"
    / "p6b_model_complexity"
    / "btumqa_225k_clean_metadata_task_trained_parameter_audit"
)
TABLES_DIR = PHASE06B_DIR / "tables"
REPORTS_DIR = PHASE06B_DIR / "reports"
DONE_DIR = PHASE06B_DIR / "done"
for path in [TABLES_DIR, REPORTS_DIR, DONE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

OUTPUT_PATHS = {
    "audit": TABLES_DIR / "phase06b_task_trained_parameter_audit.csv",
    "table_vi": TABLES_DIR / "phase06b_table_vi_parameter_values.csv",
    "components": TABLES_DIR / "phase06b_component_parameter_breakdown.csv",
    "availability": REPORTS_DIR / "phase06b_source_artifact_availability.csv",
    "report": REPORTS_DIR / "phase06b_parameter_audit_report.json",
    "done": DONE_DIR / "phase06b_parameter_audit_complete.json",
}

print("Project Drive directory:", PROJECT_DRIVE_DIR)
print("Phase 06B output directory:", PHASE06B_DIR)

Mounted at /content/drive
Project Drive directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging
Phase 06B output directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_06b_model_complexity/btumqa_225k_task_trained_parameter_audit


## Safe I/O and Counting Helpers


In [ ]:
def now_string():
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(tmp, path)


def save_dataframe(frame: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)
    print("Saved", path, "rows=", len(frame))


def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def count_model_parameters(model: nn.Module):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    return {"total": int(total), "trainable": int(trainable), "frozen": int(total - trainable)}


def count_state_dict_tensors(state_dict: dict):
    return int(sum(value.numel() for value in state_dict.values() if torch.is_tensor(value)))


def component_breakdown(model: nn.Module, method: str):
    rows = []
    for name, child in model.named_children():
        count = sum(parameter.numel() for parameter in child.parameters())
        rows.append({"method": method, "component": name, "parameters": int(count)})
    return rows


## Exact Evaluated Answer-Model Architectures


In [ ]:
class AUGRAnswerBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        model_dim = 512
        self.question_proj = nn.Sequential(
            nn.Linear(768, model_dim), nn.LayerNorm(model_dim), nn.GELU(), nn.Dropout(0.20)
        )
        self.visual_proj = nn.Sequential(nn.Linear(512, model_dim), nn.LayerNorm(model_dim))
        self.aux_encoder = nn.Sequential(
            nn.Linear(4, 64), nn.GELU(), nn.Linear(64, model_dim)
        )
        self.question_region_aux_gate = nn.Sequential(
            nn.Linear(model_dim * 2, model_dim),
            nn.LayerNorm(model_dim),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(model_dim, 1),
        )
        self.region_embeddings = nn.Parameter(torch.randn(6, model_dim) * 0.02)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=model_dim, num_heads=8, dropout=0.20, batch_first=True
        )
        self.attention_fusion_norm = nn.LayerNorm(model_dim)
        self.classifier = nn.Sequential(
            nn.Linear(model_dim * 6, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(512, 23),
        )


class QuestionImageMLP(nn.Module):
    def __init__(self, image_dim: int):
        super().__init__()
        hidden_dim = 768
        self.image_proj = nn.Sequential(
            nn.LayerNorm(image_dim), nn.Linear(image_dim, hidden_dim), nn.GELU(), nn.Dropout(0.25)
        )
        self.question_proj = nn.Sequential(
            nn.LayerNorm(768), nn.Linear(768, hidden_dim), nn.GELU(), nn.Dropout(0.25)
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim, 23),
        )


MODEL_SPECS = [
    {
        "method": "BiomedCLIP",
        "method_detail": "Frozen dual-view matching",
        "model_key": "biomedclip_global23_dual_view_naturalized",
        "model_factory": None,
        "expected_count": 0,
        "source_notebook": "BrainTumorVQA_Phase_05D_A3_(BTUMQA_225K_BiomedCLIP_Dual_View_Naturalized_Global23_Ensemble).ipynb",
    },
    {
        "method": "BiomedCLIP",
        "method_detail": "Question-image MLP",
        "model_key": "biomedclip_frozen_dual_view_question_image_mlp",
        "model_factory": lambda: QuestionImageMLP(1024),
        "expected_count": 2_582_551,
        "source_notebook": "BrainTumorVQA_Phase_05D_A4_(BTUMQA_225K_BiomedCLIP_Frozen_Feature_Supervised_Baseline).ipynb",
    },
    {
        "method": "DenseNet121",
        "method_detail": "Question-image MLP",
        "model_key": "densenet121_dual_view_question_image_mlp",
        "model_factory": lambda: QuestionImageMLP(2048),
        "expected_count": 3_371_031,
        "source_notebook": "BrainTumorVQA_Phase_05D_B1_(BTUMQA_225K_CNN_Frozen_Feature_Supervised_Baselines).ipynb",
    },
    {
        "method": "ResNet50",
        "method_detail": "Question-image MLP",
        "model_key": "resnet50_dual_view_question_image_mlp",
        "model_factory": lambda: QuestionImageMLP(4096),
        "expected_count": 4_947_991,
        "source_notebook": "BrainTumorVQA_Phase_05D_B1_(BTUMQA_225K_CNN_Frozen_Feature_Supervised_Baselines).ipynb",
    },
    {
        "method": "EfficientNet-B0",
        "method_detail": "Question-image MLP",
        "model_key": "efficientnet_b0_dual_view_question_image_mlp",
        "model_factory": lambda: QuestionImageMLP(2560),
        "expected_count": 3_765_271,
        "source_notebook": "BrainTumorVQA_Phase_05D_B1_(BTUMQA_225K_CNN_Frozen_Feature_Supervised_Baselines).ipynb",
    },
    {
        "method": "ViT-B/16",
        "method_detail": "Question-image MLP",
        "model_key": "vit_b_16_dual_view_question_image_mlp",
        "model_factory": lambda: QuestionImageMLP(1536),
        "expected_count": 2_976_791,
        "source_notebook": "BrainTumorVQA_Phase_05D_C1_(BTUMQA_225K_Attention_Frozen_Feature_Supervised_Baselines).ipynb",
    },
    {
        "method": "Swin-T",
        "method_detail": "Question-image MLP",
        "model_key": "swin_t_dual_view_question_image_mlp",
        "model_factory": lambda: QuestionImageMLP(1536),
        "expected_count": 2_976_791,
        "source_notebook": "BrainTumorVQA_Phase_05D_C1_(BTUMQA_225K_Attention_Frozen_Feature_Supervised_Baselines).ipynb",
    },
    {
        "method": "AUGR-VQA",
        "method_detail": "QAdp-DG-PRUGTM answer backbone",
        "model_key": "qadp_dg_prugtm",
        "model_factory": AUGRAnswerBackbone,
        "expected_count": 3_859_288,
        "source_notebook": "BrainTumorVQA_Phase_04_(BTUMQA_225K_QGCA_QAdp_PRUGTM_Hybrid_Clean_Metadata_Four_Seeds).ipynb",
    },
]


## Drive Artifact Registry


In [ ]:
SEEDS = [42, 1337, 2025, 3407]

PHASE4_DIR = (
    PROJECT_DRIVE_DIR
    / "phase_4_qgca_training"
    / "clean_metadata_training"
    / "adaptive_prugtm_qgca_btumqa_four_seeds"
)
PHASE5D_BASE = PROJECT_DRIVE_DIR / "phase_5d_modern_baseline_comparison_models"
A4_DIR = PHASE5D_BASE / "vlm_comparison" / "phase05d_a4_biomedclip_frozen_feature_supervised_baseline"
B1_DIR = PHASE5D_BASE / "cnn_model_comparison" / "phase05d_b1_cnn_frozen_feature_supervised_baselines"
C1_DIR = PHASE5D_BASE / "attention_model_comparison" / "phase05d_c1_attention_frozen_feature_supervised_baselines"
A3_DIR = (
    PHASE5D_BASE
    / "vlm_comparison"
    / "phase05d_a3_biomedclip_global23_dual_view_naturalized"
)


def checkpoint_paths_for(model_key: str):
    if model_key == "qadp_dg_prugtm":
        return [
            PHASE4_DIR / f"seed_{seed}" / "checkpoints" / "best_adaptive_prugtm_qgca_model.pt"
            for seed in SEEDS
        ]
    if model_key.startswith("biomedclip_frozen"):
        base = A4_DIR
    elif model_key.startswith(("densenet", "resnet", "efficientnet")):
        base = B1_DIR
    elif model_key.startswith(("vit", "swin")):
        base = C1_DIR
    else:
        return []
    return [
        base / "checkpoints" / model_key / f"seed_{seed}" / "best_model.pt"
        for seed in SEEDS
    ]


A3_PROVENANCE_ARTIFACTS = [
    A3_DIR / "predictions" / "test_predictions.csv",
    A3_DIR / "reports" / "biomedclip_global23_dual_view_naturalized_full_inference_report.json",
    A3_DIR / "done" / "biomedclip_global23_dual_view_naturalized_full_complete.json",
]


## Reconstruct and Verify Counts


In [ ]:
audit_rows = []
availability_rows = []
component_rows = []

for spec in MODEL_SPECS:
    if spec["model_factory"] is None:
        reconstruction_count = 0
        reconstruction_ok = True
        checkpoint_counts = []
        a3_dir_exists = A3_DIR.is_dir()
        availability_rows.append({
            "method": spec["method"], "model_key": spec["model_key"],
            "artifact_type": "A3 manually organized result directory",
            "path": str(A3_DIR), "exists": a3_dir_exists,
        })
        for path in A3_PROVENANCE_ARTIFACTS:
            availability_rows.append({
                "method": spec["method"], "model_key": spec["model_key"],
                "artifact_type": "A3 optional provenance artifact",
                "path": str(path), "exists": path.exists(),
            })
        seeds_verified = "not_applicable_frozen_inference"
        artifact_ok = a3_dir_exists
        checkpoint_count = np.nan
    else:
        model = spec["model_factory"]()
        counts = count_model_parameters(model)
        reconstruction_count = counts["trainable"]
        reconstruction_ok = reconstruction_count == spec["expected_count"]
        component_rows.extend(component_breakdown(model, spec["method"] + " | " + spec["method_detail"]))

        checkpoint_counts = []
        verified_seeds = []
        for seed, path in zip(SEEDS, checkpoint_paths_for(spec["model_key"])):
            exists = path.exists()
            availability_rows.append({
                "method": spec["method"], "model_key": spec["model_key"],
                "artifact_type": f"best checkpoint seed_{seed}", "path": str(path), "exists": exists,
            })
            if exists:
                payload = safe_torch_load(path)
                state_dict = payload.get("model_state_dict", payload)
                count = count_state_dict_tensors(state_dict)
                checkpoint_counts.append(count)
                verified_seeds.append(seed)
        seeds_verified = ",".join(map(str, verified_seeds))
        artifact_ok = len(verified_seeds) == len(SEEDS)
        checkpoint_count = checkpoint_counts[0] if checkpoint_counts else np.nan

    checkpoints_consistent = (
        True if not checkpoint_counts else len(set(checkpoint_counts)) == 1
    )
    checkpoint_matches = (
        True if not checkpoint_counts else checkpoint_counts[0] == reconstruction_count
    )
    verification_ok = bool(
        reconstruction_ok and artifact_ok and checkpoints_consistent and checkpoint_matches
    )

    audit_rows.append({
        "method": spec["method"],
        "method_detail": spec["method_detail"],
        "model_key": spec["model_key"],
        "parameter_scope": "task-trained answer-prediction parameters",
        "exact_task_trained_parameters": int(reconstruction_count),
        "task_trained_parameters_m": round(reconstruction_count / 1_000_000, 3),
        "source_notebook": spec["source_notebook"],
        "checkpoint_seeds_verified": seeds_verified,
        "checkpoint_parameter_count": checkpoint_count,
        "reconstruction_parameter_count": int(reconstruction_count),
        "verification_status": "verified" if verification_ok else "failed",
        "notes": (
            "Frozen matching; no task-specific optimization."
            if spec["model_factory"] is None
            else "Frozen encoders and offline feature extractors excluded."
        ),
    })

audit_df = pd.DataFrame(audit_rows)
availability_df = pd.DataFrame(availability_rows)
components_df = pd.DataFrame(component_rows)
display(audit_df)


,method,method_detail,model_key,parameter_scope,exact_task_trained_parameters,task_trained_parameters_m,source_notebook,checkpoint_seeds_verified,checkpoint_parameter_count,reconstruction_parameter_count,verification_status,notes
0,BiomedCLIP,Frozen dual-view matching,biomedclip_global23_dual_view_naturalized,task-trained answer-prediction parameters,0,0.000,BrainTumorVQA_Phase_05D_A3_(BTUMQA_225K_Biomed...,not_applicable_frozen_inference,NaN,0,verified,Frozen matching; no task-specific optimization.
1,BiomedCLIP,Question-image MLP,biomedclip_frozen_dual_view_question_image_mlp,task-trained answer-prediction parameters,2582551,2.583,BrainTumorVQA_Phase_05D_A4_(BTUMQA_225K_Biomed...,"42,1337,2025,3407",2582551.0,2582551,verified,Frozen encoders and offline feature extractors...
2,DenseNet121,Question-image MLP,densenet121_dual_view_question_image_mlp,task-trained answer-prediction parameters,3371031,3.371,BrainTumorVQA_Phase_05D_B1_(BTUMQA_225K_CNN_Fr...,"42,1337,2025,3407",3371031.0,3371031,verified,Frozen encoders and offline feature extractors...
3,ResNet50,Question-image MLP,resnet50_dual_view_question_image_mlp,task-trained answer-prediction parameters,4947991,4.948,BrainTumorVQA_Phase_05D_B1_(BTUMQA_225K_CNN_Fr...,"42,1337,2025,3407",4947991.0,4947991,verified,Frozen encoders and offline feature extractors...
4,EfficientNet-B0,Question-image MLP,efficientnet_b0_dual_view_question_image_mlp,task-trained answer-prediction parameters,3765271,3.765,BrainTumorVQA_Phase_05D_B1_(BTUMQA_225K_CNN_Fr...,"42,1337,2025,3407",3765271.0,3765271,verified,Frozen encoders and offline feature extractors...
5,ViT-B/16,Question-image MLP,vit_b_16_dual_view_question_image_mlp,task-trained answer-prediction parameters,2976791,2.977,BrainTumorVQA_Phase_05D_C1_(BTUMQA_225K_Attent...,"42,1337,2025,3407",2976791.0,2976791,verified,Frozen encoders and offline feature extractors...
6,Swin-T,Question-image MLP,swin_t_dual_view_question_image_mlp,task-trained answer-prediction parameters,2976791,2.977,BrainTumorVQA_Phase_05D_C1_(BTUMQA_225K_Attent...,"42,1337,2025,3407",2976791.0,2976791,verified,Frozen encoders and offline feature extractors...
7,AUGR-VQA,QAdp-DG-PRUGTM answer backbone,qadp_dg_prugtm,task-trained answer-prediction parameters,3859288,3.859,BrainTumorVQA_Phase_04_(BTUMQA_225K_QGCA_QAdp_...,"42,1337,2025,3407",3859288.0,3859288,verified,Frozen encoders and offline feature extractors...


## Save Authoritative Tables and Completion Report


In [ ]:
save_dataframe(availability_df, OUTPUT_PATHS["availability"])
save_dataframe(audit_df, OUTPUT_PATHS["audit"])
save_dataframe(components_df, OUTPUT_PATHS["components"])

table_vi_df = audit_df[
    ["method", "method_detail", "exact_task_trained_parameters", "task_trained_parameters_m", "verification_status"]
].copy()
save_dataframe(table_vi_df, OUTPUT_PATHS["table_vi"])

failures = audit_df.loc[audit_df["verification_status"] != "verified", "model_key"].tolist()
report = {
    "created_at": now_string(),
    "phase": "Phase 06B",
    "notebook": "BrainTumorVQA_Phase_06B_(BTUMQA_225K_Task_Trained_Parameter_Audit).ipynb",
    "parameter_scope": "task-trained answer-prediction parameters",
    "exclusions": [
        "frozen pretrained encoders",
        "offline feature extractors",
        "Q-CUR post-prediction reliability estimator",
    ],
    "no_training_or_inference": True,
    "all_rows_verified": not failures,
    "failed_model_keys": failures,
    "outputs": {key: str(value) for key, value in OUTPUT_PATHS.items()},
    "table_vi_rows": table_vi_df.to_dict(orient="records"),
}
write_json(OUTPUT_PATHS["report"], report)

if failures:
    raise RuntimeError(
        "Phase 06B verification failed for: " + ", ".join(failures)
        + ". Availability and audit reports were saved; no done marker was written."
    )

write_json(OUTPUT_PATHS["done"], report)
print("Phase 06B parameter audit completed successfully.")
print("Table VI source:", OUTPUT_PATHS["table_vi"])
print("Done marker:", OUTPUT_PATHS["done"])


Saved /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_06b_model_complexity/btumqa_225k_task_trained_parameter_audit/reports/phase06b_source_artifact_availability.csv rows= 32
Saved /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_06b_model_complexity/btumqa_225k_task_trained_parameter_audit/tables/phase06b_task_trained_parameter_audit.csv rows= 8
Saved /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_06b_model_complexity/btumqa_225k_task_trained_parameter_audit/tables/phase06b_component_parameter_breakdown.csv rows= 25
Saved /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_06b_model_complexity/btumqa_225k_task_trained_parameter_audit/tables/phase06b_table_vi_parameter_values.csv rows= 8
Phase 06B parameter audit completed successfully.
Table VI source: 